# Section 6 – Distribution and Statistical Comparison (Revised)

Analyses score distributions, rank distributions, descriptive statistics, score quartiles and category comparisons.

**Revisions:** reusable loader/filter; score histogram marks mean and median; replaces the uninformative exact-rank histogram with a score-versus-rank plot; quartile cut-points are reported and ties are handled without forcing false equal-sized groups; category comparisons display sample sizes and are stratified by scoring-comparison group to avoid mixing incompatible weighting schemes. Cross-category differences remain descriptive unless the same scoring basis is used.

##  Import Required Libraries

We use Pandas for data handling, NumPy for numerical operations, and Matplotlib for visualizations.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

def load_master(path=None):
    """Load the NIRF master CSV from a supplied path or common Colab/local locations."""
    candidates = ([Path(path)] if path else []) + [
        Path("nirf_master_final.csv"),
        Path("/content/nirf_master_final.csv"),
        Path("/mnt/data/nirf_master_final.csv")
    ]
    for candidate in candidates:
        if candidate.exists():
            data = pd.read_csv(candidate)
            print(f"Loaded: {candidate} | shape={data.shape}")
            return data
    raise FileNotFoundError(
        "nirf_master_final.csv not found. Upload it to Colab or pass its full path to load_master(path)."
    )

def filter_data(data, **filters):
    """Reusable exact-value filters; e.g. filter_data(df, year=2025, category='Engineering')."""
    result = data.copy()
    for column, value in filters.items():
        if column not in result.columns:
            raise KeyError(f"Filter column '{column}' is not present.")
        result = result[result[column] == value]
    return result.copy()

df = load_master()

Loaded: /mnt/data/nirf_master_final.csv | shape=(11012, 42)


## Load the NIRF Dataset

We load the final cleaned dataset containing institutional rankings, scores, categories, and parameter values.

In [2]:
# Dataset is loaded by load_master() above.
print("Rows:", len(df), "| Columns:", len(df.columns))
display(df.head())

Rows: 11012 | Columns: 42


,institution_key,institute_name,institute_id,institute_name_original,year,category,category_original,category_group,city,city_original,state,state_code,state_original,rank,rank_value,rank_status,rank_original,rank_low,rank_high,is_rank_band,rank_band_label,rank_band_source,rank_midpoint,score,TLR,RPC,GO,OI,PR,weight_scheme,is_top10,is_top50,is_top100,is_new_entry,first_year_in_category,category_first_year,categories_in_year,source_page,weight_scheme_basis,has_score,has_components,score_comparison_group
0,INST0936,Miranda House,IR17-COLL-1-29115,Miranda House,2017,College,College,General,Delhi,Delhi,Delhi,DL,Delhi,1.0,1.0,exact,1,1.0,1.0,False,NaN,NaN,1.0,69.39,75.76,27.44,90.06,96.45,41.82,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10
1,INST0814,Loyola College,IR17-COLL-2-959129,Loyola College,2017,College,College,General,Chennai,District Chennai,Tamil Nadu,TN,Tamil Nadu,2.0,2.0,exact,2,2.0,2.0,False,NaN,NaN,2.0,68.68,66.92,100.00,48.42,56.45,80.45,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10
2,INST1313,Shri Ram College of Commerce,IR17-COLL-2-29134,Shri Ram College of Commerce,2017,College,College,General,Delhi,Delhi,Delhi,DL,Delhi,3.0,3.0,exact,3,3.0,3.0,False,NaN,NaN,3.0,67.18,77.63,0.20,97.51,93.80,53.98,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10
3,INST0172,Bishop Heber College,IR17-COLL-1-19387,Bishop Heber College,2017,College,College,General,Tiruchirappalli,District Tiruchirapalli,Tamil Nadu,TN,Tamil Nadu,4.0,4.0,exact,4,4.0,4.0,False,NaN,NaN,4.0,61.18,76.62,53.99,61.28,68.11,18.63,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10
4,INST0093,Atma Ram Sanatan Dharma College,IR17-COLL-2-21799,Atma Ram Sanatan Dharma College,2017,College,College,General,New Delhi,New Delhi,Delhi,DL,Delhi,5.0,5.0,exact,5,5.0,5.0,False,NaN,NaN,5.0,60.68,74.22,30.54,79.60,82.51,0.21,30/20/25/15/10,True,True,True,False,2017,2017,6,CollegeRanking,estimated_from_data,True,True,College | 30/20/25/15/10


##  Understand the Dataset

Before performing statistical analysis, we examine the columns, data types, years, categories, and rank status values.

In [3]:
print("Dataset information:")
df.info()

print("\nAvailable years:")
print(sorted(df["year"].dropna().unique()))

print("\nAvailable categories:")
print(sorted(df["category"].dropna().unique()))

print("\nRank status values:")
print(df["rank_status"].value_counts(dropna=False))

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False))

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11012 entries, 0 to 11011
Data columns (total 42 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   institution_key          11012 non-null  object 
 1   institute_name           11012 non-null  object 
 2   institute_id             6966 non-null   object 
 3   institute_name_original  11012 non-null  object 
 4   year                     11012 non-null  int64  
 5   category                 11012 non-null  object 
 6   category_original        11012 non-null  object 
 7   category_group           11012 non-null  object 
 8   city                     11012 non-null  object 
 9   city_original            11002 non-null  object 
 10  state                    11012 non-null  object 
 11  state_code               11012 non-null  object 
 12  state_original           11012 non-null  object 
 13  rank                     6954 non-null   float64
 14  r

rank_band_source           6966
rank_band_label            6966
TLR                        4339
GO                         4339
PR                         4339
OI                         4339
RPC                        4339
score                      4089
rank                       4058
rank_original              4056
institute_id               4046
score_comparison_group      292
weight_scheme               292
weight_scheme_basis         292
rank_midpoint                12
rank_high                    12
rank_low                     12
rank_value                   12
city_original                10
category_group                0
category_original             0
category                      0
institute_name_original       0
year                          0
institution_key               0
institute_name                0
city                          0
is_rank_band                  0
state                         0
rank_status                   0
state_original                0
state_co

##  Select the Year and Category

We select a particular year and category for the first four questions.

The values below can be changed whenever we want to perform the analysis for another year or category.

In [4]:
selected_year = 2025
selected_category = "Engineering"

print("Selected year:", selected_year)
print("Selected category:", selected_category)

Selected year: 2025
Selected category: Engineering


##  Prepare the Data for Analysis

We create a reusable function to prepare the selected category and year.

Only institutions with exact ranks are included. Missing score and rank values are not replaced with zero.

In [5]:
def prepare_category_year_data(data, year, category):
    result = filter_data(data, year=year, category=category)
    result = result[result["rank_status"].astype(str).str.strip().str.lower().eq("exact")].copy()
    result["rank"] = pd.to_numeric(result["rank"], errors="coerce")
    result["score"] = pd.to_numeric(result["score"], errors="coerce")
    result = result.dropna(subset=["institution_key","rank"])
    if result.duplicated("institution_key").any():
        print("Warning: duplicate institution keys found; keeping the row with the most complete score data.")
        result["_ok"] = result["score"].notna().astype(int)
        result = result.sort_values("_ok").drop_duplicates("institution_key", keep="last").drop(columns="_ok")
    return result

filtered_data = prepare_category_year_data(df, selected_year, selected_category)
print("Filtered exact-ranked institutions:", len(filtered_data))
display(filtered_data.head())

Filtered exact-ranked institutions: 100


,institution_key,institute_name,institute_id,institute_name_original,year,category,category_original,category_group,city,city_original,state,state_code,state_original,rank,rank_value,rank_status,rank_original,rank_low,rank_high,is_rank_band,rank_band_label,rank_band_source,rank_midpoint,score,TLR,RPC,GO,OI,PR,weight_scheme,is_top10,is_top50,is_top100,is_new_entry,first_year_in_category,category_first_year,categories_in_year,source_page,weight_scheme_basis,has_score,has_components,score_comparison_group
9955,INST0585,Indian Institute of Technology Madras,IR-E-U-0456,Indian Institute of Technology Madras,2025,Engineering,Engineering,Discipline,Chennai,Chennai,Tamil Nadu,TN,Tamil Nadu,1.0,1.0,exact,1,1.0,1.0,False,NaN,NaN,1.0,88.72,95.70,90.74,82.29,63.25,100.00,30/30/20/10/10,True,True,True,False,2017,2017,16,EngineeringRanking,estimated_from_data,True,True,Engineering | 30/30/20/10/10
9956,INST0574,Indian Institute of Technology Delhi,IR-E-I-1074,Indian Institute of Technology Delhi,2025,Engineering,Engineering,Discipline,New Delhi,New Delhi,Delhi,DL,Delhi,2.0,2.0,exact,2,2.0,2.0,False,NaN,NaN,2.0,85.74,85.89,93.47,81.67,62.01,93.97,30/30/20/10/10,True,True,True,False,2017,2017,16,EngineeringRanking,estimated_from_data,True,True,Engineering | 30/30/20/10/10
9957,INST0573,Indian Institute of Technology Bombay,IR-E-U-0306,Indian Institute of Technology Bombay,2025,Engineering,Engineering,Discipline,Mumbai,Mumbai,Maharashtra,MH,Maharashtra,3.0,3.0,exact,3,3.0,3.0,False,NaN,NaN,3.0,83.65,87.97,84.81,87.06,59.13,85.04,30/30/20/10/10,True,True,True,False,2017,2017,16,EngineeringRanking,estimated_from_data,True,True,Engineering | 30/30/20/10/10
9958,INST0583,Indian Institute of Technology Kanpur,IR-E-I-1075,Indian Institute of Technology Kanpur,2025,Engineering,Engineering,Discipline,Kanpur,Kanpur,Uttar Pradesh,UP,Uttar Pradesh,4.0,4.0,exact,4,4.0,4.0,False,NaN,NaN,4.0,81.82,91.77,76.34,88.08,59.18,78.49,30/30/20/10/10,True,True,True,False,2017,2017,16,EngineeringRanking,estimated_from_data,True,True,Engineering | 30/30/20/10/10
9959,INST0584,Indian Institute of Technology Kharagpur,IR-E-U-0573,Indian Institute of Technology Kharagpur,2025,Engineering,Engineering,Discipline,Kharagpur,Kharagpur,West Bengal,WB,West Bengal,5.0,5.0,exact,5,5.0,5.0,False,NaN,NaN,5.0,78.69,85.26,77.56,79.82,64.24,74.57,30/30/20/10/10,True,True,True,False,2017,2017,16,EngineeringRanking,estimated_from_data,True,True,Engineering | 30/30/20/10/10


# Q1: Study the Distribution of NIRF Scores

In this question, we examine how NIRF scores are distributed among institutions in the selected category and year.

We use a histogram to understand the frequency of institutions across different score ranges.

Only exact-ranked institutions with non-missing scores are included.

## Prepare the Score Data

We extract the available NIRF scores and calculate the number of institutions included in the analysis.

In [6]:
score_data = filtered_data["score"].dropna()

print("Number of institutions with available scores:",
      len(score_data))

print("Minimum score:", score_data.min())
print("Maximum score:", score_data.max())

Number of institutions with available scores: 100
Minimum score: 45.55
Maximum score: 88.72


### Score histogram with mean and median

The vertical reference lines make the distribution's centre visible. Skewness is reported so that the distribution is not described from the histogram alone.

In [7]:
score_data = filtered_data["score"].dropna()
if score_data.empty:
    print("No valid scores for this selection.")
else:
    mean_score, median_score = score_data.mean(), score_data.median()
    skew_score = score_data.skew()
    fig = px.histogram(filtered_data.dropna(subset=["score"]), x="score", nbins=15,
                       title=f"Distribution of NIRF scores – {selected_category} ({selected_year})",
                       labels={"score":"NIRF score"})
    fig.add_vline(x=mean_score, line_dash="dash", line_color="red",
                  annotation_text=f"Mean {mean_score:.2f}")
    fig.add_vline(x=median_score, line_dash="dot", line_color="green",
                  annotation_text=f"Median {median_score:.2f}")
    fig.show()
    print(f"Mean={mean_score:.2f}; Median={median_score:.2f}; Skewness={skew_score:.3f}")
    print("Positive skew indicates a longer right tail; negative skew indicates a longer left tail.")

Mean=56.82; Median=55.59; Skewness=1.254
Positive skew indicates a longer right tail; negative skew indicates a longer left tail.


# Q2: Examine rank position against score

A histogram of exact ranks 1–100 is not informative when the ranking contains one institution at nearly every rank. This question therefore uses a **score-versus-rank scatter plot**, which shows whether higher scores are associated with better (numerically smaller) ranks.

##  Prepare the Rank Data

We extract valid institutional ranks and calculate basic information about their distribution.

In [8]:
rank_data = filtered_data["rank"].dropna()

print("Number of institutions:", len(rank_data))
print("Best rank:", rank_data.min())
print("Lowest rank:", rank_data.max())

Number of institutions: 100
Best rank: 1.0
Lowest rank: 100.0


## Visualize the Rank Distribution

The histogram shows how many institutions fall within different ranking ranges.

In [9]:
rank_score_data = filtered_data.dropna(subset=["rank","score"]).copy()
fig = px.scatter(rank_score_data, x="rank", y="score", hover_name="institute_name",
                 title=f"NIRF score versus exact rank – {selected_category} ({selected_year})",
                 labels={"rank":"Exact rank (lower is better)","score":"NIRF score"})
fig.update_xaxes(autorange="reversed")
fig.show()
print("Institutions included:", len(rank_score_data))

Institutions included: 100


# Q3: Calculate Statistical Measures of NIRF Scores

In this question, we calculate important descriptive statistical measures.

These include:

1. Mean – Average score of institutions.
2. Median – Middle score after arranging the values.
3. Standard deviation – Measures how spread out the scores are.
4. Minimum – Lowest score.
5. Maximum – Highest score.

Only exact-ranked institutions with available scores are considered.

##  Calculate Statistical Measures

In [10]:
score_statistics = {
    "Number of Institutions": score_data.count(),
    "Mean Score": score_data.mean(),
    "Median Score": score_data.median(),
    "Standard Deviation": score_data.std(),
    "Minimum Score": score_data.min(),
    "Maximum Score": score_data.max(),
    "Score Range": score_data.max() - score_data.min()
}

score_summary = pd.DataFrame(
    score_statistics.items(),
    columns=["Statistical Measure", "Value"]
)

display(score_summary)


,Statistical Measure,Value
0,Number of Institutions,100.000000
1,Mean Score,56.818500
2,Median Score,55.595000
3,Standard Deviation,9.396976
4,Minimum Score,45.550000
5,Maximum Score,88.720000
6,Score Range,43.170000


##  Interpret the Statistical Results

The mean and median describe the central tendency of institutional scores.

The standard deviation describes how much the scores vary from the average.

A smaller standard deviation indicates that scores are relatively close to one another, while a larger standard deviation indicates greater variation.

# Q4: Score quartiles using score cut-points

Quartiles are calculated from the score distribution, not by first ranking rows to force exactly equal counts. If many institutions share the same score, quantile boundaries may coincide; `duplicates='drop'` preserves tied scores instead of arbitrarily splitting ties. Therefore, group sizes may not be exactly 25% each.

## Create Quartile Groups

In [11]:
quartile_data = filtered_data[["institution_key","institute_name","score"]].dropna(subset=["score"]).copy()
if quartile_data["score"].nunique() < 2:
    raise ValueError("Not enough distinct score values to form score quartiles.")
quartile_data["Quartile"] = pd.qcut(quartile_data["score"], q=4, duplicates="drop")
quartile_summary = (quartile_data.groupby("Quartile", observed=False)["score"]
                    .agg(Number_of_Institutions="count", Minimum_Score="min", Maximum_Score="max")
                    .reset_index())
quartile_summary["Percentage"] = 100*quartile_summary["Number_of_Institutions"]/len(quartile_data)
quartile_summary["Quartile"] = quartile_summary["Quartile"].astype(str)
display(quartile_summary)

,Quartile,Number_of_Institutions,Minimum_Score,Maximum_Score,Percentage
0,"(45.549, 49.202]",25,45.55,49.03,25.0
1,"(49.202, 55.595]",25,49.26,55.28,25.0
2,"(55.595, 62.138]",25,55.91,62.08,25.0
3,"(62.138, 88.72]",25,62.31,88.72,25.0


## Calculate the Number of Institutions in Each Group

We calculate the number and percentage of institutions in each quartile.

In [12]:
# The quartile summary above includes actual score boundaries and group sizes.
quartile_summary

,Quartile,Number_of_Institutions,Minimum_Score,Maximum_Score,Percentage
0,"(45.549, 49.202]",25,45.55,49.03,25.0
1,"(49.202, 55.595]",25,49.26,55.28,25.0
2,"(55.595, 62.138]",25,55.91,62.08,25.0
3,"(62.138, 88.72]",25,62.31,88.72,25.0


##  Visualize the Quartile Distribution

A bar chart helps us compare the number of institutions in each score group.

In [ ]:
# Plot the actual score interval for each quartile, with Q1, median and Q3 cut-points.
quartile_plot = quartile_data.copy()
quartile_plot["Quartile"] = pd.qcut(quartile_plot["score"], q=4, duplicates="drop").astype(str)
quartile_cutpoints = (
    quartile_plot.groupby("Quartile", observed=True)["score"]
    .agg(Minimum="min", Q1=lambda x:x.quantile(.25), Median="median",
         Q3=lambda x:x.quantile(.75), Maximum="max", Institutions="count")
    .reset_index()
)
quartile_cutpoints["Quartile"] = pd.Categorical(
    quartile_cutpoints["Quartile"], categories=quartile_summary["Quartile"].tolist(), ordered=True
)
quartile_cutpoints = quartile_cutpoints.sort_values("Quartile")
fig = go.Figure()
for _, row in quartile_cutpoints.iterrows():
    label = str(row["Quartile"])
    fig.add_trace(go.Scatter(
        x=[row["Minimum"], row["Maximum"]], y=[label, label],
        mode="lines", line={"width":12}, name=label,
        hovertemplate=(f"{label}<br>Min: {row['Minimum']:.2f}<br>Q1: {row['Q1']:.2f}"
                       f"<br>Median: {row['Median']:.2f}<br>Q3: {row['Q3']:.2f}"
                       f"<br>Max: {row['Maximum']:.2f}<br>Institutions: {int(row['Institutions'])}<extra></extra>"),
        showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=[row["Q1"], row["Median"], row["Q3"]], y=[label]*3,
        mode="markers", marker={"size":[10,13,10], "symbol":["line-ns","diamond","line-ns"]},
        hovertemplate="Cut-point: %{x:.2f}<extra></extra>", showlegend=False
    ))
fig.update_layout(
    title=f"Actual score ranges and quartile cut-points – {selected_category} ({selected_year})",
    xaxis_title="NIRF score", yaxis_title="Score quartile",
    yaxis={"categoryorder":"array","categoryarray":quartile_summary["Quartile"].tolist()},
    template="plotly_white"
)
fig.show()


# Q5: Compare categories on a comparable scoring basis

NIRF categories may use different weighting schemes. Do not rank categories by raw mean score when their scoring bases differ. This table groups observations by both category and `score_comparison_group` (and reports `weight_scheme` where available), includes unique institution counts, and presents descriptive statistics. Compare category rows only within a common comparison group. Research or other categories without a common comparison group remain separate.

##  Prepare Data for All Categories

In [14]:
category_data = df[df["year"] == selected_year].copy()
category_data["score"] = pd.to_numeric(category_data["score"], errors="coerce")
category_data = category_data[
    category_data["rank_status"].astype(str).str.strip().str.lower().eq("exact")
].dropna(subset=["category","score","institution_key"])
category_data = category_data.drop_duplicates(["category","institution_key"])
if "score_comparison_group" not in category_data:
    category_data["score_comparison_group"] = "unspecified"
if "weight_scheme" not in category_data:
    category_data["weight_scheme"] = "unspecified"
print("Unique institutions represented:", category_data[["category","institution_key"]].drop_duplicates().shape[0])
display(category_data[["category","score_comparison_group","weight_scheme","score"]].head())

Unique institutions represented: 910


,category,score_comparison_group,weight_scheme,score
9636,Agriculture,Agriculture | 30/30/20/10/10,30/30/20/10/10,89.23
9637,Agriculture,Agriculture | 30/30/20/10/10,30/30/20/10/10,75.87
9638,Agriculture,Agriculture | 30/30/20/10/10,30/30/20/10/10,75.83
9639,Agriculture,Agriculture | 30/30/20/10/10,30/30/20/10/10,68.95
9640,Agriculture,Agriculture | 30/30/20/10/10,30/30/20/10/10,67.66


##  Calculate Category-wise Statistical Measures

We group institutions by category and calculate descriptive statistics for their NIRF scores.

In [ ]:
category_statistics = (
    category_data.groupby(["weight_scheme","category"], dropna=False)
    .agg(n=("institution_key","nunique"), Mean=("score","mean"), Median=("score","median"),
         Minimum=("score","min"), Q1=("score",lambda x:x.quantile(.25)),
         Q3=("score",lambda x:x.quantile(.75)), Maximum=("score","max"),
         Standard_Deviation=("score","std"))
    .reset_index()
)
category_statistics["IQR"] = category_statistics["Q3"] - category_statistics["Q1"]
category_statistics = category_statistics.sort_values(
    ["weight_scheme","Median"], ascending=[True,False]
).round(2)
display(category_statistics)
print("Compare categories only within the same weight_scheme.")


## Visualize Category-wise Score Distributions

A boxplot is used to compare the score distributions across different categories.

It displays the median, spread, and possible outliers for each category.

In [ ]:
# Compare categories by their actual weight scheme, not by score_comparison_group.
category_data["weight_scheme"] = category_data["weight_scheme"].fillna("Unspecified").astype(str).str.strip()
weight_scheme_order = category_data["weight_scheme"].value_counts().index.tolist()
fig = px.box(
    category_data, x="category", y="score", color="category",
    facet_col="weight_scheme", facet_col_wrap=2,
    category_orders={"weight_scheme":weight_scheme_order},
    hover_name="institute_name",
    hover_data={"weight_scheme":True,"score":":.2f","category":True},
    points="outliers",
    title=f"Category score distributions grouped by weight scheme ({selected_year})",
    labels={"category":"NIRF category","score":"NIRF score","weight_scheme":"Weight scheme"}
)
fig.update_xaxes(tickangle=35)
fig.update_layout(showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("weight_scheme=", "Weights: ")))
fig.show()


# Final Summary – Section 6

In this section, we performed distribution analysis and statistical comparison using the NIRF dataset.

The five questions helped us understand:

1. How NIRF scores are distributed among institutions.
2. How institutions are spread across ranking positions.
3. The mean, median, standard deviation, minimum, and maximum scores.
4. How institutions are distributed across four score groups.
5. How score distributions differ across NIRF categories.

Histograms helped us understand score and rank distributions.

Descriptive statistics provided numerical summaries of institutional performance.

Quartile analysis helped divide institutions into four approximately equal-sized groups.

Finally, category-wise statistical analysis and boxplots helped us compare score distributions across different NIRF categories.

Overall, these statistical techniques provide a better understanding of institutional performance and ranking patterns.

In [17]:
print("="*60)
print("SECTION 6 – DISTRIBUTION AND STATISTICAL COMPARISON (REVISED)")
print("="*60)
print("Selection:", selected_category, selected_year)
print("Q1 score count:", len(score_data))
print("Q2 score-rank observations:", len(rank_score_data))
print("\nQ3 descriptive statistics:")
display(score_summary)
print("\nQ4 quartile score ranges:")
display(quartile_summary)
print("\nQ5 category comparison by scoring group:")
display(category_statistics)
print("Note: raw scores from different scoring schemes must not be directly ranked.")

SECTION 6 – DISTRIBUTION AND STATISTICAL COMPARISON (REVISED)
Selection: Engineering 2025
Q1 score count: 100
Q2 score-rank observations: 100

Q3 descriptive statistics:


,Statistical Measure,Value
0,Number of Institutions,100.000000
1,Mean Score,56.818500
2,Median Score,55.595000
3,Standard Deviation,9.396976
4,Minimum Score,45.550000
5,Maximum Score,88.720000
6,Score Range,43.170000



Q4 quartile score ranges:


,Quartile,Number_of_Institutions,Minimum_Score,Maximum_Score,Percentage
0,"(45.549, 49.202]",25,45.55,49.03,25.0
1,"(49.202, 55.595]",25,49.26,55.28,25.0
2,"(55.595, 62.138]",25,55.91,62.08,25.0
3,"(62.138, 88.72]",25,62.31,88.72,25.0



Q5 category comparison by scoring group:


,score_comparison_group,category,weight_scheme,n,Mean,Median,Minimum,Q1,Q3,Maximum,Standard_Deviation,IQR
0,Agriculture | 30/30/20/10/10,Agriculture,30/30/20/10/10,40,57.16,55.50,46.87,49.88,60.63,89.23,9.18,10.75
1,Architecture | 40/20/20/10/10,Architecture,40/20/20/10/10,40,58.77,57.46,47.61,53.88,60.82,83.95,8.06,6.94
2,College | 40/15/25/10/10,College,40/15/25/10/10,100,62.06,59.66,54.39,56.84,65.07,84.01,7.33,8.22
3,Dental | 30/30/20/10/10,Dental,30/30/20/10/10,39,60.73,57.42,52.19,55.04,62.51,89.12,8.35,7.47
4,Engineering | 30/30/20/10/10,Engineering,30/30/20/10/10,100,56.82,55.60,45.55,49.20,62.14,88.72,9.40,12.94
5,Law | 40/15/25/10/10,Law,40/15/25/10/10,40,62.32,60.47,52.70,55.68,65.31,82.97,8.54,9.63
6,Management | 30/30/20/10/10,Management,30/30/20/10/10,100,56.19,54.06,44.53,49.03,60.41,83.29,9.31,11.38
7,Medical | 30/30/20/10/10,Medical,30/30/20/10/10,50,61.12,59.81,50.91,54.54,64.84,91.80,8.45,10.31
8,Overall | 30/30/20/10/10,Overall,30/30/20/10/10,100,57.76,55.35,49.04,51.98,60.49,87.31,8.18,8.51
9,Pharmacy | 30/30/20/10/10,Pharmacy,30/30/20/10/10,101,54.12,52.60,41.48,44.24,60.89,86.59,10.89,16.65


Note: raw scores from different scoring schemes must not be directly ranked.
